# 🗺️ Project Pulse — The Music Atlas

A **generic, open** map of music built from sound — *not* from anyone's personal taste.
We use a public, Creative-Commons dataset (**FMA**, the Free Music Archive) with real
**genre** labels, and view the same songs through different **lenses**:

- **Sound lens** — a deep neural audio fingerprint (AST → UMAP)
- **Feature lens** — plain acoustic features (brightness, tempo, texture…)
- **Genre** — the colour overlay on both

The fun part: a song's *genre* can scatter across the *sound* map — genre and sound
don't agree. Then we overlay one person's Liked/Disliked songs to see where a real
listener falls in the broader atlas.

> Inspired by maps like *Every Noise at Once*, but open-source, song-level, and
> multi-lens. Built on Colab's free GPU. **Runtime ▸ Run all**, then sit back ~15–25 min
> for the one-time build.

> Honest note: even FMA skews Western/indie/electronic — it is NOT a complete picture of
> global music. That bias is part of the story, not hidden.

## Step 1 — Setup

In [1]:
#@title Install (torch is already on Colab; don't reinstall it)
!pip -q install umap-learn librosa transformers plotly 2>/dev/null
import torch
print("GPU available:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("If this says CPU only: Runtime ▸ Change runtime type ▸ GPU, then Run all again.")

GPU available: True | Tesla T4
If this says CPU only: Runtime ▸ Change runtime type ▸ GPU, then Run all again.


## Step 2 — Load the engine + the personal-taste overlay

This clones the project to reuse `pulse_core.py` and the 99 personal songs.
**Set `REPO_URL` to your GitHub repo** after you publish it.

In [2]:
#@title Get the project code + data
REPO_URL = "https://github.com/mdmmirfan/Project_Pulse.git"  #@param {type:"string"}
import os
if not os.path.exists("Project_Pulse"):
    !git clone -q $REPO_URL Project_Pulse
%cd Project_Pulse
import pulse_core, pandas as pd, numpy as np
print("Engine loaded.")

/content/Project_Pulse
Engine loaded.


## Step 3 — Download the FMA dataset (subset)

`fma_metadata.zip` (~342 MB) has genres; `fma_small.zip` (~7 GB) has 30-second clips.
Colab's network is fast, so this is usually a few minutes.

In [3]:
#@title Download + unzip FMA
import os
BASE = "https://os.unil.cloud.switch.ch/fma"
for name in ["fma_metadata.zip", "fma_small.zip"]:
    if not os.path.exists(name):
        print("Downloading", name, "...")
        !wget -q $BASE/$name
        !unzip -q -o $name
print("FMA ready:", os.path.exists("fma_metadata/tracks.csv"), os.path.exists("fma_small"))

FMA ready: True True


## Step 4 — Pick a balanced subset (N songs per genre)

In [4]:
#@title Sample the subset
N_PER_GENRE   = 40   #@param {type:"integer"}
SNIPPET_SECONDS = 30 #@param {type:"integer"}

tracks = pd.read_csv("fma_metadata/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[("set", "subset")] == "small"]
genre_top = small[("track", "genre_top")].dropna()

def fma_path(tid):
    s = f"{tid:06d}"
    return os.path.join("fma_small", s[:3], s + ".mp3")

rng = np.random.default_rng(42)
sample = []
for g in sorted(genre_top.unique()):
    ids = genre_top[genre_top == g].index.to_numpy()
    rng.shuffle(ids)
    for tid in ids[:N_PER_GENRE]:
        p = fma_path(tid)
        if os.path.exists(p):
            sample.append({"track_id": tid, "genre": g, "path": p})

sample = pd.DataFrame(sample)
print(f"Sampled {len(sample)} tracks across {sample['genre'].nunique()} genres:")
print(sample["genre"].value_counts().to_string())

Sampled 320 tracks across 8 genres:
genre
Electronic       40
Experimental     40
Folk             40
Hip-Hop          40
Instrumental     40
International    40
Pop              40
Rock             40


## Step 5 — Extract features for the atlas (the slow part, GPU-accelerated)

In [5]:
#@title Extract AST embedding + interpretable features for each clip
from pulse_core import extract_neural_embedding, extract_interpretable_features, load_snippet, INTERPRETABLE_FEATURES

rows = []
for i, r in sample.iterrows():
    try:
        y, sr = load_snippet(r["path"], seconds=SNIPPET_SECONDS)
        emb = extract_neural_embedding(y, sr)
        feats = extract_interpretable_features(y, sr)
        row = {"track_id": r["track_id"], "genre": r["genre"]}
        row.update({f"dim_{k}": emb[k] for k in range(len(emb))})
        row.update(feats)
        rows.append(row)
    except Exception as e:
        pass
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(sample)} done")

atlas = pd.DataFrame(rows)
print(f"\nAtlas built: {len(atlas)} tracks.")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

The image processor of type `ASTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  25/320 done
  50/320 done
  75/320 done
  100/320 done
  125/320 done
  150/320 done
  175/320 done
  200/320 done
  225/320 done
  250/320 done
  275/320 done
  300/320 done

Atlas built: 320 tracks.


## Step 6 — Build the two lenses (UMAP)

In [6]:
#@title Fit the Sound lens and Feature lens
import umap
from sklearn.preprocessing import StandardScaler

dim_cols = [c for c in atlas.columns if c.startswith("dim_")]

# Sound lens: deep AST embedding
sound_scaler = StandardScaler().fit(atlas[dim_cols])
sound_reducer = umap.UMAP(n_components=3, random_state=42).fit(sound_scaler.transform(atlas[dim_cols]))
atlas[["SX", "SY", "SZ"]] = sound_reducer.embedding_

# Feature lens: interpretable features
feat_scaler = StandardScaler().fit(atlas[INTERPRETABLE_FEATURES])
feat_reducer = umap.UMAP(n_components=3, random_state=42).fit(feat_scaler.transform(atlas[INTERPRETABLE_FEATURES]))
atlas[["FX", "FY", "FZ"]] = feat_reducer.embedding_

print("Both lenses fitted.")

Both lenses fitted.


## Step 7 — The Atlas, two ways (coloured by genre)

In [7]:
#@title Sound lens
import plotly.express as px
px.scatter_3d(atlas, x="SX", y="SY", z="SZ", color="genre",
              title="The Music Atlas — SOUND lens (AST embedding), coloured by genre",
              template="plotly_dark").update_traces(
              marker=dict(size=4, opacity=0.8)).show()

In [8]:
#@title Feature lens
px.scatter_3d(atlas, x="FX", y="FY", z="FZ", color="genre",
              title="The Music Atlas — FEATURE lens (brightness/tempo/texture), coloured by genre",
              template="plotly_dark").update_traces(
              marker=dict(size=4, opacity=0.8)).show()

## Step 8 — Overlay: where does one person's taste fall?

We project the 99 personal Liked/Disliked songs into the atlas using the **same** fitted
lenses. (Note: those embeddings were computed on full tracks, not 30s clips, so treat the
overlay as approximate.)

In [9]:
#@title Project the personal songs onto the atlas
neural = pd.read_csv("project_pulse_neural_data.csv")
interp = pd.read_csv("project_pulse_interpretable_features.csv")
me = neural.merge(interp, on=["Track", "Preference"], how="inner")

me_sound = sound_reducer.transform(sound_scaler.transform(me[dim_cols]))
me_feat  = feat_reducer.transform(feat_scaler.transform(me[INTERPRETABLE_FEATURES]))
me["SX"], me["SY"], me["SZ"] = me_sound[:, 0], me_sound[:, 1], me_sound[:, 2]
me["FX"], me["FY"], me["FZ"] = me_feat[:, 0], me_feat[:, 1], me_feat[:, 2]

# Combine atlas (by genre) + personal songs (as two highlighted classes) for one legend
atlas_plot = atlas.assign(group=atlas["genre"])
me_plot = me.assign(group="★ My " + me["Preference"])
combined = pd.concat([
    atlas_plot[["SX", "SY", "SZ", "group"]],
    me_plot[["SX", "SY", "SZ", "group"]],
], ignore_index=True)

fig = px.scatter_3d(combined, x="SX", y="SY", z="SZ", color="group",
                    title="Sound lens — FMA genres + one listener's taste overlaid",
                    template="plotly_dark")
fig.update_traces(marker=dict(size=4, opacity=0.7))
fig.show()

## Step 9 — (Optional) Drop in YOUR songs

In [10]:
#@title Upload audio and see where you land
RUN = False  #@param {type:"boolean"}
if RUN:
    from google.colab import files
    from pathlib import Path
    up = files.upload()
    paths = []
    for nm in up:
        Path(nm).write_bytes(up[nm]); paths.append(nm)

    yours = []
    for p in paths:
        y, sr = load_snippet(p, seconds=SNIPPET_SECONDS)
        emb = extract_neural_embedding(y, sr)
        c = sound_reducer.transform(sound_scaler.transform([emb]))[0]
        yours.append({"Track": Path(p).stem, "SX": c[0], "SY": c[1], "SZ": c[2]})
    yours = pd.DataFrame(yours)

    cmb = pd.concat([
        atlas.assign(group=atlas["genre"])[["SX","SY","SZ","group"]],
        yours.assign(group="★ YOUR song")[["SX","SY","SZ","group"]],
    ], ignore_index=True)
    px.scatter_3d(cmb, x="SX", y="SY", z="SZ", color="group",
                  title="Where your songs land in the atlas",
                  template="plotly_dark").update_traces(
                  marker=dict(size=4, opacity=0.7)).show()
    display(yours)
else:
    print("Set RUN = True to upload your own songs.")

Set RUN = True to upload your own songs.


## Step 10 — Save the atlas (so you don't rebuild it)

In [11]:
#@title Save artifacts + download
import joblib, os
os.makedirs("atlas_artifacts", exist_ok=True)
joblib.dump(sound_scaler,  "atlas_artifacts/sound_scaler.joblib")
joblib.dump(sound_reducer, "atlas_artifacts/sound_reducer.joblib")
joblib.dump(feat_scaler,   "atlas_artifacts/feat_scaler.joblib")
joblib.dump(feat_reducer,  "atlas_artifacts/feat_reducer.joblib")
atlas.to_csv("atlas_artifacts/atlas.csv", index=False)
!zip -q -r atlas_artifacts.zip atlas_artifacts
print("Saved. Download atlas_artifacts.zip from the file browser (left), or:")
try:
    from google.colab import files
    files.download("atlas_artifacts.zip")
except Exception:
    pass

Saved. Download atlas_artifacts.zip from the file browser (left), or:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### What you're seeing
Notice how a single genre's songs **spread out** across the sound map, and how the
feature map rearranges everything. That's the point: **genre is a label, sound is a
measurement, and they don't line up.** A song's "category" tells you less about how it
actually sounds than people assume — which is exactly why taste (and recommendation) is
so finicky. 🎶